Exploratory Data Analysis - Content Refresh Dataset
<br>

Project- Forecasting future content growth and recovery using historical using Historical Search Intelligence signals

</br>


In [1]:
# importing libraries and setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

# display settings
pd.set_option('display.max_columns',100)
pd.set_option('display.width',160)
pd.set_option('display.float_format', lambda x: f"{x:,.3f}")

sns.set_style('whitegrid')
plt.rcParams['figure.dpi']=110
%matplotlib inline

# CSV path
CSV_PATH="../../data/raw/content_refresh_anonymized.csv"

1. data overview 

In [2]:
#loading
df=pd.read_csv(CSV_PATH)

print("SHAPE (rows,columns):",df.shape)

print("\n ---column names & Dtypes---")
print(df.dtypes)

SHAPE (rows,columns): (30000, 44)

 ---column names & Dtypes---
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d 

In [3]:
df.head(3).T

,0,1,2
content_id,content_304f48230142,content_a1fb4e703a9e,content_9aa793d4d895
client_id,client_f369cb89fc,client_4e07408562,client_7f2253d7e2
search_volume,10.000,90.000,0.000
competition,0.670,0.010,0.000
competition_level,HIGH,LOW,LOW
cpc,2.050,0.050,0.000
content_type,keyword article,keyword article,keyword article
main_intent,transactional,informational,informational
word_count,"3,221.000","2,481.000","3,515.000"
char_count,"20,457.000","15,562.000","23,643.000"


In [4]:
# missing values- count + percentage, sorted by how bad they are
missing=df.isnull().sum()
missing_pct=(missing / len(df) * 100).round(2)
missing_df=pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df=missing_df[missing_df["missing_count"] > 0].sort_values("missing_pct", ascending=False)
missing_df

,missing_count,missing_pct
provider_used,21438,71.460
word_count_tier,7699,25.660
char_count,7699,25.660
word_count,7699,25.660
char_count_tier,7699,25.660
model_used,5733,19.110
trend_pct,3388,11.290
competition_level,2610,8.700
search_volume,2468,8.230
competition,2468,8.230


In [5]:
#duplicate check: full row duplicates and duplicate ids
print('Full row duplicates:', df.duplicated().sum())
print('duplicate content_id:', df['content_id'].duplicated().sum())
print('unique content_id:', df['content_id'].nunique())
print('unique client id:', df['client_id'].nunique())

Full row duplicates: 0
duplicate content_id: 0
unique content_id: 30000
unique client id: 32


2. Descriptive Statistics

> Getting central tendency, spread, and range for every numeric column and frequency counts for every categorical column.

In [6]:
# spliting columns into numeric vs categorical buckets

numeric_cols=df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols=df.select_dtypes(include=['object']).columns.tolist()

print('Numerical columns:', numeric_cols)
print('\n Categorical columns:', categorical_cols)

Numerical columns: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

 Categorical columns: ['content_id', 'client_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']


In [7]:
desc=df[numeric_cols].describe().T
desc['skew']=df[numeric_cols].skew()
desc['missing_pct']=(df[numeric_cols].isnull().sum() / len(df)*100).round(2)
desc

,count,mean,std,min,25%,50%,75%,max,skew,missing_pct
search_volume,"27,532.000",158.882,"1,518.271",0.000,0.000,10.000,20.000,"74,000.000",26.016,8.230
competition,"27,532.000",0.147,0.285,0.000,0.000,0.000,0.130,1.000,2.044,8.230
cpc,"27,532.000",0.485,2.102,0.000,0.000,0.000,0.000,100.360,13.738,8.230
word_count,"22,301.000","3,107.760","1,452.383",8.000,"2,413.000","2,877.000","3,666.000","9,546.000",0.938,25.660
char_count,"22,301.000","20,665.278","10,115.344",40.000,"15,644.000","19,116.000","24,011.000","111,158.000",1.551,25.660
impressions_90d,"30,000.000","5,200.366","16,838.020",1.000,81.000,731.000,"3,615.250","517,715.000",11.385,0.000
clicks_90d,"30,000.000",16.097,75.077,0.000,0.000,1.000,7.000,"4,178.000",18.346,0.000
pageviews_90d,"30,000.000",49.942,152.101,0.000,2.000,8.000,33.000,"5,998.000",10.861,0.000
sessions_90d,"30,000.000",37.067,107.069,1.000,2.000,7.000,27.000,"4,345.000",12.127,0.000
users_90d,"30,000.000",35.938,103.748,1.000,2.000,7.000,27.000,"4,913.000",13.101,0.000


In [8]:
# categorical value counts(skipping content_id/client_id, these are ids not categories)
for col in categorical_cols:
    if col in ("content_id", "client_id"):
        continue
    vc = df[col].value_counts(dropna=False)
    vc_pct = (vc/len(df)*100).round(1)
    print(f"\n{col} (n_unique={df[col].nunique()}):")
    display(pd.DataFrame({"count": vc, "pct": vc_pct}))


competition_level (n_unique=3):


,count,pct
competition_level,,
LOW,22896,76.300
HIGH,2658,8.900
NaN,2610,8.700
MEDIUM,1836,6.100



content_type (n_unique=3):


,count,pct
content_type,,
keyword article,27207,90.700
feedly article,2096,7.000
comparison article,697,2.300



main_intent (n_unique=4):


...

,count,pct
impression_tier,,
low,11248,37.500
moderate,10469,34.900
good,7205,24.000
excellent,1078,3.600



position_tier (n_unique=5):


,count,pct
position_tier,,
page_1,11814,39.400
striking,7304,24.300
page_3_5,7242,24.100
top_3,2321,7.700
deep,1319,4.400



trend_direction (n_unique=5):


,count,pct
trend_direction,,
down,16262,54.200
stable,5962,19.900
up,4388,14.600
new,2236,7.500
flat,1152,3.800


3. Target variable deep dive(trend_pct)

> For this forecasting project, trend_pct and its categorical sibling trend_direction is the key target.

In [9]:
print('---trend_pct basic stats---')
print(df['trend_pct'].describe())

---trend_pct basic stats---
count   26,612.000
mean        -4.786
std        473.862
min       -100.000
25%        -62.600
50%        -33.500
75%          0.000
max     44,900.000
Name: trend_pct, dtype: float64


In [11]:
df.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,29950,29951,29952,29953,29954,29955,29956,29957,29958,29959,29960,29961,29962,29963,29964,29965,29966,29967,29968,29969,29970,29971,29972,29973,29974,29975,29976,29977,29978,29979,29980,29981,29982,29983,29984,29985,29986,29987,29988,29989,29990,29991,29992,29993,29994,29995,29996,29997,29998,29999
content_id,content_304f48230142,content_a1fb4e703a9e,content_9aa793d4d895,content_331d6c4de07b,content_d99b7a2d90ca,content_d4084a4bc775,content_9a34b442b552,content_a63219c6e95a,content_5e6c160719bc,content_c27558df2b0c,content_d8ee6cc6d642,content_5a3e876cf7f7,content_42fb2cad9ecf,content_a5a2fbc76336,content_91067a14431a,content_689414059706,content_78bd1d4a1d4d,content_761a44afda12,content_0b360eb9db55,content_af865035b328,content_0d748c484ab1,content_9d548144b06d,content_3fb46bec4413,content_2da6ae9d0882,content_0e23e310d404,content_033ae3e7aecf,content_72c5c2d73e5a,content_7ea135180dd9,content_19ad8f9bac29,content_ba8e51f13800,content_249298388b45,content_24ee79621dbf,content_5eeba5d398f2,content_d87a116e2c79,content_55f75c034970,content_1a28b25c7128,content_bce275871a25,content_724a2783045d,content_dbe82879a406,content_4595e8704e07,content_4df0b7207fe3,content_e45a618f6b32,content_3223b1640f5a,content_1938955b34c4,content_793b7376a0e5,content_2a6383ed421f,content_c59d46264834,content_40cb4af260c0,content_326fa2fa449f,content_f0717373e86e,...,content_7d52e164c23c,content_b018d113bdaf,content_e6cc2aad65ea,content_ac21791a5807,content_5daabd0e31f3,content_a31cad37ea8a,content_20decd85a0c2,content_3b806fcc5b2c,content_d6ce4e17f464,content_73cf70f08e06,content_b1d45033b059,content_326a540b3a6e,content_be106cd29636,content_7ba9b154acf6,content_07ea0872b973,content_a3af3b8346d8,content_77867ed726e1,content_b9e02bd01a73,content_179533212cd0,content_b5db993ce36a,content_4998a1c76243,content_92a5d2709aa9,content_1db0d204d42f,content_4be930227848,content_b00f10211e25,content_bdee2164f576,content_851c604b0631,content_c87291853cab,content_3dc420aa9809,content_6f2f3043b633,content_81a91fe32bc2,content_2dfd17269502,content_fe5d259e6bc5,content_6880eb215048,content_a6568a7c07d7,content_fcad8c75dc44,content_677c622c0bfb,content_7b36d6ffe9bd,content_9bb9a0584cae,content_e859812ce999,content_b7fa2d76e5ae,content_04a7a1b7b5ec,content_23dce6a656e6,content_9bd30342fd4a,content_995627a1f490,content_c322796023c8,content_526572edb3fa,content_38112bdd0c6e,content_ab26273a7e7a,content_887020f20b5e
client_id,client_f369cb89fc,client_4e07408562,client_7f2253d7e2,client_19581e27de,client_3fdba35f04,client_f369cb89fc,client_8722616204,client_19581e27de,client_6208ef0f77,client_19581e27de,client_19581e27de,client_d4735e3a26,client_6208ef0f77,client_8527a891e2,client_6208ef0f77,client_9f14025af0,client_6208ef0f77,client_19581e27de,client_349c41201b,client_f369cb89fc,client_f369cb89fc,client_6208ef0f77,client_4e07408562,client_e629fa6598,client_19581e27de,client_f369cb89fc,client_4e07408562,client_4ec9599fc2,client_3fdba35f04,client_d59eced1de,client_2c624232cd,client_19581e27de,client_bbb965ab0c,client_19581e27de,client_d029fa3a95,client_6208ef0f77,client_f369cb89fc,client_4e07408562,client_6208ef0f77,client_8527a891e2,client_9f14025af0,client_7f2253d7e2,client_a88a7902cb,client_f369cb89fc,client_8527a891e2,client_7f2253d7e2,client_19581e27de,client_f369cb89fc,client_98a3ab7c34,client_8527a891e2,...,client_8527a891e2,client_7f2253d7e2,client_19581e27de,client_19581e27de,client_6208ef0f77,client_8b940be7fb,client_6208ef0f77,client_6208ef0f77,client_f369cb89fc,client_19581e27de,client_e29c9c180c,client_8527a891e2,client_d029fa3a95,client_19581e27de,client_2c624232cd,client_25fc0e7096,client_19581e27de,client_2c624232cd,client_f74efabef1,client_6208ef0f77,client_f369cb89fc,client_25fc0e7096,client_6208ef0f77,client_d4735e3a26,client_3fdba35f04,client_6208ef0f77,client_6208ef0f77,client_d029fa3a95,client_7f2253d7e2,clie

In [13]:
"""Reverse engineer the formula behind the trend_pct by reconstructing it from sessions, clicks, and impressions separately, then checking which one matches exactly. this is crucial before building forecasting features"""

tmp=df.copy()

tmp["recon_sessions"]=np.where(tmp["sessions_prev_30d"] > 0, (tmp['sessions_last_30d'] - tmp['sessions_prev_30d']) / tmp['sessions_prev_30d'] * 100, np.nan)

tmp["recon_clicks"] = np.where(tmp["clicks_prev_30d"] > 0,(tmp["clicks_last_30d"] - tmp["clicks_prev_30d"]) / tmp["clicks_prev_30d"] * 100,np.nan,
)

tmp["recon_impressions"] = np.where(tmp["impressions_prev_30d"] > 0,(tmp["impressions_last_30d"] - tmp["impressions_prev_30d"]) / tmp["impressions_prev_30d"] * 100,np.nan,
)

for label, col in [("sessions","recon_sessions"),("clicks","recon_clicks"),("impressions","recon_impressions")]:
    comp = tmp[["trend_pct",col]].dropna()
    diff = (comp['trend_pct'] - comp[col]).abs()
    match_rate = (diff < 1.0).mean()*100
    print(f"reconstructed from {label:12s} -> match rate within 1.0: {match_rate:6.1f}% (n={len(comp)})")


reconstructed from sessions     -> match rate within 1.0:    4.1% (n=21873)
reconstructed from clicks       -> match rate within 1.0:    1.4% (n=11759)
reconstructed from impressions  -> match rate within 1.0:  100.0% (n=26612)


# from the above observation 
> we can conclude that trend_pct is calculated from IMPRESSIONS (100% exact match) 
>
> i used trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100
>
> this means trend_pct reflects SEARCH VISIBILITY change, not actual
>


In [14]:
# why is trend_pct missing for 11% of rows? checking if its a division by zero issue

missing_trend = df[df['trend_pct'].isnull()]
print('rows with missing trend_pct:', len(missing_trend))
print(missing_trend[['clicks_prev_30d', 'sessions_prev_30d','impressions_prev_30d']].describe())
print("\n% of missing-trend rows where impressions_prev_30d == 0:",
(missing_trend["impressions_prev_30d"] == 0).mean() * 100)

rows with missing trend_pct: 3388
       clicks_prev_30d  sessions_prev_30d  impressions_prev_30d
count        3,388.000          3,388.000             3,388.000
mean             0.000              0.743                 0.000
std              0.000              2.084                 0.000
min              0.000              0.000                 0.000
25%              0.000              0.000                 0.000
50%              0.000              0.000                 0.000
75%              0.000              1.000                 0.000
max              0.000             69.000                 0.000

% of missing-trend rows where impressions_prev_30d == 0: 100.0


> there are around 3388 rows which has null value in trend_pct feature